# PrivateDFL — a walkthrough

Companion notebook for *Privacy-Preserving Decentralized Federated Learning via Explainable
Adaptive Differential Privacy* (arXiv:2509.10691).

Three things make this framework work, and we look at each in turn: the hyperdimensional
encoder, the noise accountant, and the serverless ring that ties them together.

**Setup** — from the repository root:

```bash
pip install -e ".[all]"
```

The preprocessed UCI-HAR data ships in `Dataset/`, so nothing needs downloading.

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import torch

import privatedfl as pdfl
from privatedfl.utils import configure_logging

configure_logging(logging.WARNING)   # keep the notebook quiet
print("privatedfl", pdfl.__version__, "| torch", torch.__version__)
print("datasets found:", pdfl.available_datasets("Dataset"))

---
## 1. The data

`load_dataset` reads the `.choir_dat` files, L2-normalises each row, and splits the training
set across the ring. Under `non-IID` every client is restricted to two activity classes.

In [ ]:
K = 100

dataset = pdfl.load_dataset("UCIHAR", n_clients=K, partition="non-IID", root="Dataset", seed=42)
print(dataset.summary())

for client in dataset.clients[:4]:
    counts = client.class_counts(dataset.n_classes)
    print(f"{client.client_id}: {len(client):4d} samples, class counts {counts}")

In [ ]:
matrix = np.stack([c.class_counts(dataset.n_classes) / len(c) for c in dataset.clients[:30]])

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(matrix, cmap="viridis", aspect="auto")
ax.set_xlabel("Activity class")
ax.set_ylabel("Client")
ax.set_title("Two classes per client (first 30 of the ring)")
fig.colorbar(image, ax=ax, label="Fraction of client's data")
plt.tight_layout(); plt.show()

---
## 2. The encoder

Encoding is Eq. (1): project through a fixed Gaussian basis, then take a cosine.

```python
basis = torch.randn(n_features, D)
H = torch.cos(X @ basis)
```

Note what is *absent*: no binarisation. Hypervectors stay real-valued in `[-1, 1]`, and that
is exactly what bounds the sensitivity at `sqrt(D)` in the privacy proofs.

In [ ]:
encoder = pdfl.Encoder(n_features=dataset.n_features, dimensions=2000, seed=42)

sample = torch.from_numpy(dataset.clients[0].x[:1])
encoded = encoder(sample)

print("input  :", tuple(sample.shape))
print("output :", tuple(encoded.shape))
print("range  : [%.3f, %.3f]" % (encoded.min(), encoded.max()))
print("distinct values: %d  (would be 2 if binarised)" % len(encoded.unique()))
print("||H||  : %.2f   sensitivity bound sqrt(D) = %.2f" % (encoded.norm(), np.sqrt(2000)))

In [ ]:
# Locality: similar inputs must encode to similar hypervectors.
anchor = dataset.clients[0].x[0]
rng = np.random.default_rng(0)
direction = rng.normal(size=anchor.shape).astype(np.float32)
direction /= np.linalg.norm(direction)

deltas = np.linspace(0, 0.5, 25)
variants = (anchor + np.outer(deltas, direction)).astype(np.float32)
hv = encoder(torch.from_numpy(variants))
similarity = (hv @ hv[0]) / encoder.dimensions

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(deltas, similarity.numpy(), marker="o")
ax.set_xlabel("Perturbation size in input space")
ax.set_ylabel("Hypervector similarity")
ax.set_title("The cosine encoding preserves locality")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 3. The noise accountant

This is the contribution. In a ring there is no aggregation, so the model's state is fully
described by how many updates have touched it:

$$t = K(r-1) + k$$

That single counter collapses the paper's four theorems into two cases:

| | Injected variance |
| --- | --- |
| $t = 1$ (Theorem 2) | $\frac{2D}{\epsilon^2}\ln\frac{1.25N}{\delta_0}$ |
| $t \geq 2$ (Theorems 3-5) | $\frac{2D}{\epsilon^2}\ln\frac{t}{t-1}$ |

Let's verify that the increments telescope exactly to the requirement.

In [ ]:
D, EPS, DELTA0 = 2000, 0.5, 1e-3
N = dataset.min_client_samples()
kwargs = dict(dimensions=D, epsilon=EPS, delta0=DELTA0, samples_per_client=N)

for t in (1, 10, 500, 3000):
    total = sum(pdfl.incremental_variance(**kwargs, step=j) for j in range(1, t + 1))
    required = pdfl.required_variance(**kwargs, step=t)
    print(f"t={t:5d}   sum of increments = {total:14.4f}   required = {required:14.4f}   "
          f"match = {np.isclose(total, required)}")

In [ ]:
steps = np.arange(1, K * 30 + 1)
tracked  = np.array([pdfl.required_variance(**kwargs, step=int(t)) for t in steps])
added    = np.array([pdfl.incremental_variance(**kwargs, step=int(t)) for t in steps])
blackbox = np.array([pdfl.blackbox_cumulative_variance(**kwargs, step=int(t)) for t in steps])

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))

left.plot(steps, tracked, lw=2, label="PrivateDFL (tracked)")
left.plot(steps, blackbox, lw=2, ls="--", label="black-box (untracked)")
left.set_yscale("log"); left.set_xlabel("Client update $t$")
left.set_ylabel("Cumulative noise variance (log)")
left.set_title("Tracking keeps growth logarithmic"); left.legend(); left.grid(alpha=0.3)

right.plot(steps, added, color="crimson", lw=2)
right.set_yscale("log"); right.set_xlabel("Client update $t$")
right.set_ylabel(r"Injected variance $\Gamma_k^r$ (log)")
right.set_title("The first client pays almost everything"); right.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"After {len(steps)} updates the untracked scheme carries "
      f"{blackbox[-1] / tracked[-1]:,.0f}x the noise variance.")

The right-hand panel explains the accuracy curve you are about to see. Step 1 injects a
variance of ~183,000 into prototypes whose components are of order 10²–10³ — the model starts
buried. By step 3000 each client adds a variance of about 5. Rounds 2 onward dig the signal
back out against a nearly frozen noise floor.

### Why `epsilon` hurts and `delta0` does not

In [ ]:
print("eps    variance         |  delta0   variance")
for eps, d0 in zip([1.0, 0.5, 0.1, 0.05], [1e-3, 1e-6, 1e-12, 1e-18]):
    v_eps = pdfl.required_variance(dimensions=D, epsilon=eps, delta0=1e-3,
                                   samples_per_client=N, step=1000)
    v_d0  = pdfl.required_variance(dimensions=D, epsilon=0.5, delta0=d0,
                                   samples_per_client=N, step=1000)
    print(f"{eps:<6} {v_eps:14,.0f}  |  {d0:<8.0e} {v_d0:12,.0f}")
print("\nVariance scales as 1/eps^2 but only as ln(1/delta0).")

---
## 4. A full run around the ring

100 clients, 30 rounds, no server. Round 1 builds prototypes and accumulates them; every later
round hands the model from client to client, each one correcting it against its own data and
perturbing it before passing it on.

In [ ]:
config = pdfl.ExperimentConfig(
    dataset="UCIHAR",
    partition="non-IID",
    n_clients=K,
    rounds=30,
    dimensions=D,
    epsilon=EPS,
    delta0=DELTA0,
    similarity="dot",
    seed=42,
    data_root="Dataset",
)

configure_logging(logging.INFO)
history = pdfl.run_privatedfl(dataset, config)
configure_logging(logging.WARNING)

print()
print(history.final_report)

In [ ]:
rounds = [r.round_index for r in history.rounds]

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))

left.plot(rounds, [a * 100 for a in history.accuracies], marker="o", color="crimson")
left.axhline(100 / dataset.n_classes, ls=":", color="gray", label="chance")
left.set_xlabel("Communication round"); left.set_ylabel("Test accuracy (%)")
left.set_title("Accuracy climbs as retraining outruns the noise")
left.legend(); left.grid(alpha=0.3)

right.plot(rounds, [r.noise_required for r in history.rounds], marker="s", label="tracked")
right.plot(rounds, [r.noise_blackbox for r in history.rounds], marker="^", ls="--",
           label="black-box")
right.set_yscale("log")
right.set_xlabel("Communication round"); right.set_ylabel("Cumulative variance (log)")
right.set_title("What the accountant saves"); right.legend(); right.grid(alpha=0.3)

plt.tight_layout(); plt.show()

---
## 5. What privacy costs

Rerun with DP disabled to get the ceiling.

In [ ]:
clear_config = pdfl.ExperimentConfig(**{**config.to_dict(), "differential_privacy": False})
clear = pdfl.run_privatedfl(dataset, clear_config, progress=False)

private_acc = history.final_report.accuracy * 100
clear_acc = clear.final_report.accuracy * 100
print(f"with DP (eps={EPS:g}) : {private_acc:.2f}%")
print(f"without DP        : {clear_acc:.2f}%")
print(f"cost of privacy   : {clear_acc - private_acc:.2f} points")

---
## 6. How much should you trust one number?

Under a tight budget the round-1 draw dominates, so a single run is a noisy estimate. Before
quoting a headline figure, look at the spread.

In [ ]:
accuracies = []
for seed in range(5):
    d = pdfl.load_dataset("UCIHAR", n_clients=K, partition="non-IID", root="Dataset", seed=seed)
    c = pdfl.ExperimentConfig(**{**config.to_dict(), "seed": seed, "eval_every": 30})
    h = pdfl.run_privatedfl(d, c, progress=False)
    accuracies.append(h.final_report.accuracy * 100)
    print(f"seed {seed}: {accuracies[-1]:.2f}%")

values = np.array(accuracies)
print(f"\nmean {values.mean():.2f}%   sd {values.std(ddof=1):.2f}   "
      f"range [{values.min():.2f}, {values.max():.2f}]")
print("Report mean +/- sd over several seeds rather than a single run.")

---
## 7. Where to go next

- `privatedfl --help` — every option from the command line
- `configs/` — paper, IID, non-private and quick configurations
- `scripts/plot_noise_accounting.py` — Section 4.2 in one command
- `scripts/sweep_privacy.py` — the epsilon x delta0 grid of Figure 5
- `scripts/seed_variance.py` — the spread analysis above, at scale
- `docs/ALGORITHM.md` — the derivations behind `privacy.py`
- `docs/REPRODUCING.md` — what differs from the released notebook, and why

Open directions the paper names: asynchronous and dynamic topologies, heterogeneous
per-client privacy budgets, robustness to poisoning and backdoors, and combining the
accountant with secure aggregation or lightweight homomorphic encryption.